# RAG Pipeline Test Playground
Use this notebook to run and test individual components of the RAG pipeline.

## 0. Setup

In [1]:
import os, sys
from pathlib import Path

# The 'rag' directory is itself the package, so its parent must be on sys.path
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in a .env file at the project root"
print("API key loaded.")

API key loaded.


## 1. Test: Parser
Parse a local file and inspect the raw parent chunks returned.

In [2]:
FILE_PATH = "dynamo.pdf"   # ← change to your file
PARSER    = "unstructured"   # "unstructured" | "docling"

In [3]:
source = Path(FILE_PATH).name

In [4]:
from unstructured.partition.auto import partition
elements = partition(
            filename=source, # Path to your PDF file
            strategy="hi_res", # Use the processing method of extraction
            extract_image_block_to_payload=True, # Store images as base64 data you can actually use
            infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
            extract_image_block_types=["Image"], # Grab images found in the PDF
        )

/Users/aditya.bansal@zomato.com/projects/rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 9368.15it/s]


In [4]:
from rag.parsers.unstructured_parser import UnstructuredParser
from rag.parsers.docling_parser import DoclingParser

parser = UnstructuredParser() if PARSER == "unstructured" else DoclingParser()
raw_parents = parser.parse(FILE_PATH)

print(f"{len(raw_parents)} parent sections found")

/Users/aditya.bansal@zomato.com/projects/rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No languages specified, defaulting to English.
Loading weights: 100%|██████████| 367/367 [00:00<00:00, 10176.98it/s]


40 parent sections found


In [5]:
from collections import Counter

for index, parent in enumerate(raw_parents):
    counts = Counter(child.chunk_type.value for child in parent.children)
    counts_str = "  ".join(f"{k}={v}" for k, v in sorted(counts.items()))
    print(f"Index:{index}  [{parent.metadata.get('section_title', '')[:50]}]  total={len(parent.children)}  {counts_str}")

Index:0  [Dynamo: Amazon’s Highly Available Key-value Store]  total=1  text=1
Index:1  [Amazon.com]  total=0  
Index:2  [ABSTRACT]  total=4  text=4
Index:3  [Categories and Subject Descriptors]  total=1  text=1
Index:4  [General Terms]  total=1  text=1
Index:5  [1. INTRODUCTION]  total=9  text=9
Index:6  [2. BACKGROUND]  total=3  text=3
Index:7  [2.1 System Assumptions and Requirements]  total=5  text=5
Index:8  [2.2 Service Level Agreements (SLA)]  total=2  image=1  text=1
Index:9  [Figure 1: Service-oriented architecture of Amazon’]  total=7  text=7
Index:10  [2.3 Design Considerations]  total=9  text=9
Index:11  [3. RELATED WORK]  total=0  
Index:12  [3.1 Peer to Peer Systems]  total=3  text=3
Index:13  [3.2 Distributed File Systems and Databases]  total=6  image=1  text=5
Index:14  [3.3 Discussion]  total=1  text=1
Index:15  [4. SYSTEM ARCHITECTURE]  total=4  table=1  text=3
Index:16  [4.1 System Interface]  total=2  text=2
Index:17  [4.2 Partitioning Algorithm]  total=7  text=7
In

In [6]:
print(raw_parents[15].children[2].raw_content)

<table><thead><tr><th>Problem</th><th></th><th>Advantage</th></tr></thead><tbody><tr><td>Partitioning</td><td>Consistent Hashing</td><td>Incremental Scalability</td></tr><tr><td>High Availability for writes</td><td>Vector clocks with reconciliation during reads</td><td>Version size is decoupled from update rates.</td></tr><tr><td>Handling temporary failures</td><td>Sloppy Quorum and hinted handoff</td><td>Provides high availability and durability guarantee when some of the replicas are not available.</td></tr><tr><td>Recovering from permanent failures</td><td>Anti-entropy using Merkle trees</td><td>Synchronizes divergent replicas in the background.</td></tr><tr><td>Membership and failure detection</td><td>Gossip-based membership protocol and failure detection.</td><td>Preserves symmetry and avoids having a centralized registry for storing membership and node liveness information.</td></tr></tbody></table>


## 2. Test: Chunker
Merge / split parent sections into child chunks.

In [7]:
from collections import Counter
from rag.chunkers.chunker import process_all_parents

def chunk_summary(chunks):
    rows = []
    for parent in chunks:
        counts = Counter(child.chunk_type.value for child in parent.children)
        rows.append((parent.id, parent.metadata.get("section_title", "")[:50], len(parent.children), counts))
    return rows

before = chunk_summary(raw_parents)

parent_chunks = process_all_parents(raw_parents)

after = chunk_summary(parent_chunks)

print(f"{'ID':34} {'Section':52} {'before':>6}  {'after':>5}  {'before_types':30}  after_types")
print("-" * 160)
for (pid, title, b_total, b_counts), (_, _, a_total, a_counts) in zip(before, after):
    b_str = "  ".join(f"{k}={v}" for k, v in sorted(b_counts.items()))
    a_str = "  ".join(f"{k}={v}" for k, v in sorted(a_counts.items()))
    print(f"{pid:34} {title:52} {b_total:>6}  {a_total:>5}  {b_str:30}  {a_str}")


ID                                 Section                                              before  after  before_types                    after_types
----------------------------------------------------------------------------------------------------------------------------------------------------------------
589eed1596cec2430f2a12bd052d78dd   Dynamo: Amazon’s Highly Available Key-value Store         1      1  text=1                          text=1
ec0f1a0157b4aa384c7d7f5d3b7711c4   Amazon.com                                                0      0                                  
d8f620b40c5f106bf9eee0934a87d161   ABSTRACT                                                  4      1  text=4                          text=1
f8a5b162a6ae572744c2b41198ee9f96   Categories and Subject Descriptors                        1      1  text=1                          text=1
5f7aacd8c4dda68fe56fce63eb9fe77e   General Terms                                             1      1  text=1                    

## 3. Test: LLM Enricher
Enrich non-text chunks (tables, images) using an LLM.

In [8]:
from rag.enrichers.enricher import LLMEnricher, build_embedding_content
from rag.models import ChunkType

# Enrich a small slice to limit API cost during testing
SAMPLE_SIZE = 3

enricher = LLMEnricher(model="gpt-4o", max_concurrency=5)

sample_parents = [
    p for p in parent_chunks
    if any(child.chunk_type in (ChunkType.IMAGE, ChunkType.TABLE) for child in p.children)
][:SAMPLE_SIZE]

print(f"Selected {len(sample_parents)} parents with image/table children")

await enricher.enrich_all(sample_parents)
build_embedding_content(sample_parents)

Selected 3 parents with image/table children
  [enricher] 8 children total: 5 text (passthrough), 3 non-text (LLM) across 3 parents
  [enriched] type=image  page=5  id=ae8005b1a33a3c2073341928c66b7601  → 'The image depicts a circular arrangement of nodes '
  [enriched] type=table  page=5  id=c5b02b0e479eec84050c7a1286dfd38a  → 'The table outlines various problems encountered in'
  [enriched] type=image  page=3  id=9fa39f467c73746ab015b45bd53c781d  → 'The image is a flowchart illustrating a system arc'
  [parent enriched] id=e10d4d56d79188abf6b439393a00fb5d  section='4. SYSTEM ARCHITECTURE'  → 'The architecture of a production-ready storage sys'
  [parent enriched] id=07e9f46ae827ddd7363c8768764613bc  section='2.2 Service Level Agreements ('  → 'The document section discusses the importance of d'
  [parent enriched] id=c5eba510fc88ca3a5fdeadda8cb79739  section='3.2 Distributed File Systems a'  → 'The document discusses various distributed storage'
  [enricher] all 3 parents summarized


In [9]:
for p in sample_parents:
    print(f"\nParent: {p.metadata.get('section_title', '')[:60]}")
    for i, child in enumerate(p.children):
        if child.chunk_type not in (ChunkType.IMAGE, ChunkType.TABLE):
            continue
        if not child.retrieved_content:
            continue
        print(f"\n  [{i}] {child.chunk_type.value}")
        print(f"  retrieved : {child.retrieved_content}")
        print(f"  embedded  : {child.embedding_content}")
        print(f"  {'─'*80}")


Parent: 2.2 Service Level Agreements (SLA)

  [1] image
  retrieved : The image is a flowchart illustrating a system architecture for handling client requests. It begins with "Client Requests" at the top, which are directed to "Page Rendering Components." These components route requests through "Request Routing" to "Aggregator Services." The requests are then further routed to various services, including "Dynamo instances," "Amazon S3," and "Other datastores." The diagram shows a hierarchical structure for processing and routing requests through different components and services, emphasizing scalability and distributed data management.
  embedded  : Section: 2.2 Service Level Agreements (SLA)
Type: image
Content: The image is a flowchart illustrating a system architecture for handling client requests. It begins with "Client Requests" at the top, which are directed to "Page Rendering Components." These components route requests through "Request Routing" to "Aggregator Services." The re

## 4. Test: Embedder
Embed a list of texts and inspect the vector shapes.

In [10]:
from rag.embedders.openai_embedder import OpenAIEmbedder

embedder = OpenAIEmbedder()   # default: text-embedding-3-small

test_texts = [
    "Revenue grew 15% year-over-year.",
    "Operating expenses increased by 8%.",
    "Net profit margin improved to 22%.",
]

vectors = embedder.embed_documents(test_texts)
print(f"Embedded {len(vectors)} texts")
print(f"Vector dimension: {len(vectors[0])}")
print(f"First vector (first 8 dims): {vectors[0][:8]}")

query_vec = embedder.embed_query("What was the revenue growth?")
print(f"\nQuery vector dim: {len(query_vec)}")

Embedded 3 texts
Vector dimension: 1536
First vector (first 8 dims): [-0.005184173583984375, 0.03564453125, 0.0010614395141601562, 0.042877197265625, 0.0304412841796875, 0.022918701171875, -0.0594482421875, 0.083984375]

Query vector dim: 1536


## 5. Test: Vector Store (Chroma)
Upsert documents and run a similarity search.

In [11]:
from langchain_core.documents import Document
from rag.vectorstores.chroma_store import ChromaVectorStore
from rag.vectorstores.base import SearchParams

vs = ChromaVectorStore(
    collection_name="rag_test",
    persist_directory="./chroma_test_db",
)

# Build a tiny synthetic dataset
docs = [
    Document(page_content=t, metadata={"source": "test", "chunk_type": "text", "chunk_id": t})
    for t in test_texts
]
vecs = embedder.embed_documents([d.page_content for d in docs])

vs.upsert(docs, vecs)
print("Upserted", len(docs), "documents")

results = vs.search(query_vector=query_vec, params=SearchParams(top_k=2))
print(f"\nSearch results ({len(results)} chunks):")
for r in results:
    print(" -", r.page_content)

Upserted 3 documents

  Chroma Dense Results  threshold=0.6
  #     Chunk ID                                    Score     Status
  ----- ---------------------------------------- -------- ----------
  0     Revenue grew 15% year-over-year.           0.6604       kept
  1     Net profit margin improved to 22%.         0.4820   filtered


Search results (1 chunks):
 - Revenue grew 15% year-over-year.


## 6. Test: Full Pipeline (end-to-end)
Run the complete RAG pipeline on a file.

In [14]:
from rag.hybrid.pipeline import HybridRAGPipeline
from rag.vectorstores.chroma_store import ChromaVectorStore
from rag.vectorstores.base import SearchParams

pipeline = HybridRAGPipeline(
    vectorstore=ChromaVectorStore(
        collection_name="rag",
        persist_directory="./chroma_test_db",
    )
)

# ── Ingest ──
parent_chunks = await pipeline.run_async(
    file_path=FILE_PATH,
    parser=PARSER,
)
print(f"Ingested {len(parent_chunks)} parent sections")

/Users/aditya.bansal@zomato.com/projects/rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# ── Hybrid search ──
QUERY = "Why does Dynamo resolve conflicts during reads rather than writes?"   # ← change this

results = await pipeline.search_async(QUERY, params=SearchParams(top_k=5, use_hybrid=True))

print(f"Top {len(results)} results for: '{QUERY}'\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] score={doc.metadata.get('score')}  type={doc.metadata.get('chunk_type')}  source={doc.metadata.get('source')}")
    print(f"     {doc.page_content[:300]}")
    print()


  [search] query='Why does Dynamo resolve conflicts during reads rather than writes?'  mode=hybrid  top_k=5

  [chroma hybrid] corpus=67 docs  top_k=5  rrf_threshold=0.02

  BM25 Rankings
  #     Chunk ID                                  RRF Score
  ----- ---------------------------------------- ----------
  0     99c9291987dfebebdceb2f26fcc61c80           0.016667
  1     c1b11dcf2feedcc9d6a2bcc60cf168aa           0.016393
  2     b67582392009f5e68121e15a32338637           0.016129
  3     4a1367e9fec4b37712b7322aa592fdd3           0.015873
  4     28333bec4b21b39c2f883f5cdfc3ebbb           0.015625
  5     c97afeab307afb6b92a5ad2404ff1cec           0.015385
  6     3d3e550472f55283b590e77ad86a3afe           0.015152
  7     33f9c41a5bf44a83f22649c5247465b7           0.014925
  8     13f720e02c0d3829bc85fe0b9bfcf10a           0.014706
  9     b3548af003687cf3f74d3be72a01f98b           0.014493

  Vector Rankings
  #     Chunk ID                                  RRF Score
  ----- ----

## 7. Test: Fusion RAG Pipeline

In [ ]:
from rag.fusion.pipeline import FusionRAGPipeline

fusion = FusionRAGPipeline(pipeline, n_queries=3)

# ── Fusion search ──
results = await fusion.search_async(QUERY, params=SearchParams(top_k=5, use_hybrid=True))

print(f"Top {len(results)} fusion results for: '{QUERY}'\n")
for i, doc in enumerate(results):
    print(f"[{i+1}] rrf_score={doc.metadata.get('rrf_score')}  type={doc.metadata.get('chunk_type')}  source={doc.metadata.get('source')}")
    print(f"     {doc.page_content[:300]}")
    print()

## 8. Test: Evaluator

In [ ]:
# HybridRAGPipeline Evaluation

from rag.evaluators.llm_evaluator import LLMEvaluator

evaluator = LLMEvaluator(pipeline.enricher.llm)  # reuses pipeline's LLM client

searcher = pipeline

print("=================hybrid search=================")
hybrid_chunks = await searcher.search_async(QUERY, params=SearchParams(top_k=5, use_hybrid=True))

print("=================dense search=================")
dense_chunks = await searcher.search_async(QUERY, params=SearchParams(top_k=5))

print(f"\nEvaluating {len(hybrid_chunks)} hybrid chunks and {len(dense_chunks)} dense chunks for query: '{QUERY}'\n")

hybrid_answer = await pipeline.generate_answer_async(QUERY, hybrid_chunks)
dense_answer = await pipeline.generate_answer_async(QUERY, dense_chunks)
print(f"\nHybrid answer:\n{hybrid_answer}\n")
print(f"\nDense answer:\n{dense_answer}\n")

hybrid_result = await evaluator.evaluate(QUERY, hybrid_chunks, hybrid_answer)
print("\nHybrid evaluation result:")
hybrid_result.print_summary()

dense_result = await evaluator.evaluate(QUERY, dense_chunks, dense_answer)
print("\nDense evaluation result:")
dense_result.print_summary()

=================hybrid search=================

  [search] query='Why does Dynamo resolve conflicts during reads rather than writes?'  mode=hybrid  top_k=5

  [chroma hybrid] corpus=67 docs  top_k=5  rrf_threshold=0.02

  BM25 Rankings
  #     Chunk ID                                  RRF Score
  ----- ---------------------------------------- ----------
  0     99c9291987dfebebdceb2f26fcc61c80           0.016667
  1     c1b11dcf2feedcc9d6a2bcc60cf168aa           0.016393
  2     b67582392009f5e68121e15a32338637           0.016129
  3     4a1367e9fec4b37712b7322aa592fdd3           0.015873
  4     28333bec4b21b39c2f883f5cdfc3ebbb           0.015625
  5     c97afeab307afb6b92a5ad2404ff1cec           0.015385
  6     3d3e550472f55283b590e77ad86a3afe           0.015152
  7     33f9c41a5bf44a83f22649c5247465b7           0.014925
  8     13f720e02c0d3829bc85fe0b9bfcf10a           0.014706
  9     b3548af003687cf3f74d3be72a01f98b           0.014493

  Vector Rankings
  #     Chunk ID        

In [21]:
# FusionRAGPipeline Evaluation

from rag.evaluators.llm_evaluator import LLMEvaluator

evaluator = LLMEvaluator(pipeline.enricher.llm)  # reuses pipeline's LLM client

searcher = fusion

print("=================hybrid search=================")
hybrid_chunks = await searcher.search_async(QUERY, params=SearchParams(top_k=5, use_hybrid=True))

print("=================dense search=================")
dense_chunks = await searcher.search_async(QUERY, params=SearchParams(top_k=5))

print(f"\nEvaluating {len(hybrid_chunks)} hybrid chunks and {len(dense_chunks)} dense chunks for query: '{QUERY}'\n")

hybrid_answer = await pipeline.generate_answer_async(QUERY, hybrid_chunks)
dense_answer = await pipeline.generate_answer_async(QUERY, dense_chunks)
print(f"\nHybrid answer:\n{hybrid_answer}\n")
print(f"\nDense answer:\n{dense_answer}\n")


hybrid_result = await evaluator.evaluate(QUERY, hybrid_chunks, hybrid_answer)
print("\nHybrid evaluation result:")
hybrid_result.print_summary()

dense_result = await evaluator.evaluate(QUERY, dense_chunks, dense_answer)
print("\nDense evaluation result:")
dense_result.print_summary()

=================hybrid search=================

  [fusion] rewritten queries (4 total):
    [original] Why does Dynamo resolve conflicts during reads rather than writes?
    [variant 1] What are the reasons for Dynamo's conflict resolution strategy being implemented during read operations instead of write operations?
    [variant 2] How does Dynamo's approach to conflict resolution during reads compare to resolving conflicts during writes?
    [variant 3] Can you explain the mechanism by which Dynamo handles conflict resolution during read operations and the benefits of this approach?

  [search] query='Why does Dynamo resolve conflicts during reads rather than writes?'  mode=hybrid  top_k=20

  [search] query='What are the reasons for Dynamo's conflict resolution strategy being implemented'  mode=hybrid  top_k=20

  [search] query='How does Dynamo's approach to conflict resolution during reads compare to resolv'  mode=hybrid  top_k=20

  [search] query='Can you explain the mechanism 

## 9. Test: Multi-Turn / Conversational RAG

When follow-up questions reference prior context (e.g. "What are its limitations?"), the `QueryContextualizer` rewrites them into fully self-contained queries before retrieval.

In [22]:
from rag.conversation.history import ConversationHistory
from rag.conversation.contextualizer import QueryContextualizer

history = ConversationHistory()

# Simulate a prior turn
history.add(
    question="How does Dynamo handle write conflicts?",
    answer="Dynamo uses vector clocks to track object versions and resolves conflicts during reads using application-defined reconciliation logic.",
)

# Follow-up that only makes sense with context
follow_up = "What are its limitations?"

contextualizer = QueryContextualizer(pipeline.enricher.llm)
standalone = await contextualizer.contextualize(follow_up, history)

print(f"Original : {follow_up}")
print(f"Rewritten: {standalone}")

Original : What are its limitations?
Rewritten: What are the limitations of using vector clocks and application-defined reconciliation logic in Dynamo for handling write conflicts?


In [ ]:
# ── Full multi-turn search loop ──
history = ConversationHistory()

questions = [
    "How does Dynamo handle write conflicts?",
    "What are its limitations?",          # ambiguous follow-up
    "How does hinted handoff solve that?", # chains on previous answer
]

for question in questions:
    print(f"\n{'='*60}")
    print(f"  Q: {question}")
    print(f"{'='*60}")

    chunks = await pipeline.search_async(question, params=SearchParams(top_k=3, use_hybrid=True), history=history)
    answer = await pipeline.generate_answer_async(question, chunks)

    print(f"\n  A: {answer}\n")
    history.add(question, answer)


  Q: How does Dynamo handle write conflicts?


AttributeError: 'HybridRAGPipeline' object has no attribute 'search_async'

## 10. Comparison: Hybrid vs Fusion × Dense vs Hybrid (multi-turn)

Runs 3 conversational questions through all 4 combinations, each with its own independent `ConversationHistory`. Evaluates every answer and prints a summary table with 4 rows per question.

In [ ]:
import asyncio
from rag.conversation.history import ConversationHistory
from rag.evaluators.llm_evaluator import LLMEvaluator
from rag.vectorstores.base import SearchParams

CONV_QUESTIONS = [
    "How does Dynamo handle write conflicts?",
    "What are its limitations?",           # ambiguous follow-up
    "How does hinted handoff solve that?",  # chained follow-up
]

MODES = {
    # "hybrid | dense":  (pipeline, SearchParams(top_k=5, use_hybrid=False)),
    # "hybrid | hybrid": (pipeline, SearchParams(top_k=5, use_hybrid=True)),
    "fusion | dense":  (fusion,   SearchParams(top_k=5, use_hybrid=False)),
    "fusion | hybrid": (fusion,   SearchParams(top_k=5, use_hybrid=True)),
}

evaluator = LLMEvaluator(pipeline.enricher.llm)
histories = {name: ConversationHistory() for name in MODES}

# results[question][mode] = (chunks, answer, EvalResult)
all_results: dict[str, dict] = {q: {} for q in CONV_QUESTIONS}

async def _run_mode(name, searcher, params, question):
    history = histories[name]
    chunks  = await searcher.search_async(question, params=params, history=history)
    answer  = await pipeline.generate_answer_async(question, chunks)
    result  = await evaluator.evaluate(question, chunks, answer)
    return name, chunks, answer, result

for question in CONV_QUESTIONS:
    print(f"\nRunning: {question[:70]}")

    # run all 4 modes in parallel (each has its own history)
    tasks = [
        _run_mode(name, searcher, params, question)
        for name, (searcher, params) in MODES.items()
    ]
    mode_outputs = await asyncio.gather(*tasks)

    for name, chunks, answer, result in mode_outputs:
        all_results[question][name] = result
        histories[name].add(question, answer)
        print(f"  [{name}] relevance={result.avg_chunk_relevance:.2f}  faithful={result.faithfulness:.2f}")

# ── Summary table ──────────────────────────────────────────────────────────
Q_W  = 42
M_W  = 16
N_W  =  7
F_W  =  9
R_W  =  9

header = f"{'Question':<{Q_W}} {'Mode':<{M_W}} {'Chunks':>{N_W}} {'Relevance':>{R_W}} {'Faithful':>{F_W}}"
sep    = "-" * len(header)

print(f"\n\n{sep}")
print(header)
print(sep)

for question in CONV_QUESTIONS:
    q_label = question[:Q_W-2]
    for i, name in enumerate(MODES):
        r      = all_results[question][name]
        q_col  = q_label if i == 0 else ""
        print(f"{q_col:<{Q_W}} {name:<{M_W}} {len(r.chunk_scores):>{N_W}} {r.avg_chunk_relevance:>{R_W}.2f} {r.faithfulness:>{F_W}.2f}")
    print(sep)


Running: How does Dynamo handle write conflicts?

  [search] query='How does Dynamo handle write conflicts?'  mode=dense  top_k=5

  Chroma Dense Results  threshold=0.6
  #     Chunk ID                                    Score     Status
  ----- ---------------------------------------- -------- ----------
  0     99c9291987dfebebdceb2f26fcc61c80           0.6587       kept
  1     caa42432732941c7601071aac61481da           0.6092       kept
  2     b67582392009f5e68121e15a32338637           0.6018       kept
  3     68b81e6c009a1d61eb8792fcacf48649           0.5992   filtered
  4     7827ef3a85390f4050b740e3dda16c61           0.5902   filtered

  [search] → 3 chunks returned

  [generate] query='How does Dynamo handle write conflicts?'  context_chunks=3

  [search] query='How does Dynamo handle write conflicts?'  mode=hybrid  top_k=5

  [chroma hybrid] corpus=67 docs  top_k=5  rrf_threshold=0.02

  BM25 Rankings
  #     Chunk ID                                  RRF Score
  ----- -----

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-4o in organization org-AfhXfDjauRFyjlqi2gIrikM2 on tokens per min (TPM): Limit 30000, Used 29349, Requested 756. Please try again in 210ms. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'tokens', 'param': None, 'code': 'rate_limit_exceeded'}}